In [1]:
import os
import gc

import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader
from transformers import PatchTSTForPrediction
from datasets import Dataset

/root/anaconda3/envs/TF/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data = "coin"

output_dir = "saved_models"
log_dir = f"logs/{data}/lstm"

loss_name = "mse"

num_train_epochs = 2000
model_num = 1
model_path = "./saved_models"
learning_rate = 1e-6

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
## target domain
target_X = pd.read_csv(f"../data/{data}/train_input_7.csv").iloc[:, 1:].values.astype(np.float32)
target_y = pd.read_csv(f"../data/{data}/train_output_7.csv").iloc[:, 1:].values.astype(np.float32)

target_X_val = target_X[-round(target_X.shape[0] * 0.2):, :].astype(np.float32)
target_y_val = target_y[-round(target_y.shape[0] * 0.2):].astype(np.float32)
target_X = target_X[:-round(target_X.shape[0] * 0.2), :].astype(np.float32)
target_y = target_y[:-round(target_y.shape[0] * 0.2)].astype(np.float32)

test_X  = pd.read_csv(f"../data/{data}/val_input_7.csv").iloc[:, 1:].values.astype(np.float32)
test_y  = pd.read_csv(f"../data/{data}/val_output_7.csv").iloc[:, 1:].values.astype(np.float32)

In [4]:
def array_to_dataset(X, y):
    X, y = torch.tensor(X), torch.tensor(y)
    X = X.reshape(-1, X.shape[1], 1)
    y = y.reshape(-1, y.shape[1], 1)

    dataset = torch.utils.data.TensorDataset(X, y)

    return dataset

train_dataset = array_to_dataset(target_X, target_y)
val_dataset = array_to_dataset(target_X_val, target_y_val)
test_dataset = array_to_dataset(test_X, test_y)

train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size = 8, shuffle = True)
val_dataloader = torch.utils.data.DataLoader(val_dataset, batch_size = 64)
test_dataloader = torch.utils.data.DataLoader(test_dataset, batch_size = 64)

In [86]:
class LSTMHead(torch.nn.Module):
    def __init__(self, iw, ow, input_dim):
        super().__init__()

        self.input_adapter = torch.nn.Linear(
            iw, ow
        )

        self.lstm1 = torch.nn.LSTM(input_dim, 128, batch_first = True)
        self.lstm2 = torch.nn.LSTM(128, 64, batch_first = True)

        self.fc = torch.nn.Linear(64, 1)

    def forward(self, x):
        ## x : (B, 1, 7, 256)
        x = x.squeeze(1)            ## (B, 7, 256)
        x = x.transpose(1, 2)       ## (B, 256, 7)
        x = self.input_adapter(x)   ## (B, 256, 24)
        x = x.transpose(1, 2)       ## (B, 24, 256)

        x, _ = self.lstm1(x)
        x, _ = self.lstm2(x)
        outputs = self.fc(x)

        return outputs

In [87]:
for k in range(1, model_num+1):
    current_path = os.path.join(model_path, f"model_{loss_name}_{k}.pth")

    model_instance = PatchTSTForPrediction.from_pretrained(os.path.join(model_path, "PatchTSTBackbone")).to(device)
    model_instance.load_state_dict(torch.load(current_path))
    
    model_instance.head.flatten = torch.nn.Identity()
    model_instance.head.projection = LSTMHead(
        iw = 7, ow = target_y.shape[1], input_dim = 256
    )
    model_instance.head.dropout = torch.nn.Identity()
    model_instance.to(device)

In [88]:
## custom loss function
def SMAPE(yhat, y):
    numerator = 100*torch.abs(y - yhat)
    denominator = (torch.abs(y) + torch.abs(yhat))/2
    smape = torch.mean(numerator / denominator)
    return smape

def MAPE(y_pred, y_true, epsilon=1e-7):
    denominator = torch.clamp(torch.abs(y_true), min=epsilon)       ## 분모에 0이 들어오는 것을 방지
    abs_percent_error = torch.abs((y_true - y_pred) / denominator)

    return torch.mean(100. * abs_percent_error)


class MASE(torch.nn.Module):
    def __init__(self, training_data, period = 1):
        super().__init__()
        ## 원본 코드 구현, 사실상 MAE와 동일, 잘못 짜여진 코드, 일단은 하던대로 할 것.
        self.scale = torch.mean(torch.abs(torch.tensor(training_data[period:] - training_data[:-period])))
    
    def forward(self, yhat, y):
        error = torch.abs(y - yhat)
        return torch.mean(error) / self.scale

In [89]:
optimizer = torch.optim.Adam(model_instance.parameters(), lr = learning_rate)
log_data = []

if loss_name == "mse":
    loss_fn = torch.nn.MSELoss()
elif loss_name == "mae":
    loss_fn = torch.nn.L1Loss()
elif loss_name == "SMAPE":
    loss_fn = SMAPE
elif loss_name == "mape":
    loss_fn = MAPE
elif loss_name == "MASE":
    loss_fn = MASE(target_y, target_y.shape[1])
else:
    raise Exception("Your loss name is not valid.")

## early stopping
PATIENCE = 10
best_val_loss = np.inf
patience_counter = 0

for epoc in range(num_train_epochs):
    model_instance.train()

    total_train_loss = 0

    for X, y in train_dataloader:
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()
        yhat = model_instance(X).prediction_outputs
        loss = loss_fn(yhat, y)
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()*X.shape[0]

    avg_train_loss = total_train_loss/len(train_dataloader.dataset)

    model_instance.eval()

    with torch.no_grad():
        yys = []
        yyhats = []

        for XX, yy in val_dataloader:
            XX = XX.to(device)
            yys.append(yy.to(device))
            yyhats.append(model_instance(XX).prediction_outputs)

        yyhat = torch.concat(yyhats)
        yy = torch.concat(yys)

        val_loss = loss_fn(yyhat, yy).item()

    print(f"Epoch {epoc+1}/{num_train_epochs} | Train Loss: {avg_train_loss:.6f}\t\t Val Loss: {val_loss:.6f}")

    log_data.append({"epoch": epoc, "loss": avg_train_loss, "eval_loss": val_loss})

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state_dict = model_instance.state_dict()   ## 저장 없이 결과물만 산출...
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= PATIENCE:
        break

/root/anaconda3/envs/TF/lib/python3.12/site-packages/torch/nn/modules/loss.py:634: UserWarning: Using a target size (torch.Size([8, 24, 1])) that is different to the input size (torch.Size([8, 1, 24])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/root/anaconda3/envs/TF/lib/python3.12/site-packages/torch/nn/modules/loss.py:634: UserWarning: Using a target size (torch.Size([2, 24, 1])) that is different to the input size (torch.Size([2, 1, 24])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/root/anaconda3/envs/TF/lib/python3.12/site-packages/torch/nn/modules/loss.py:634: UserWarning: Using a target size (torch.Size([145, 24, 1])) that is different to the input size (torch.Size([145, 1, 24])). This will likely lead to incorrect results due to broadcast

Epoch 1/2000 | Train Loss: 370.927597		 Val Loss: 28.082376
Epoch 2/2000 | Train Loss: 369.287921		 Val Loss: 27.955280
Epoch 3/2000 | Train Loss: 367.534443		 Val Loss: 27.806873
Epoch 4/2000 | Train Loss: 364.879835		 Val Loss: 27.621199
Epoch 5/2000 | Train Loss: 362.149348		 Val Loss: 27.333504
Epoch 6/2000 | Train Loss: 358.637058		 Val Loss: 26.900518
Epoch 7/2000 | Train Loss: 354.507663		 Val Loss: 26.628948
Epoch 8/2000 | Train Loss: 349.979422		 Val Loss: 26.099869
Epoch 9/2000 | Train Loss: 343.929776		 Val Loss: 25.335363
Epoch 10/2000 | Train Loss: 339.806057		 Val Loss: 25.323147
Epoch 11/2000 | Train Loss: 333.470181		 Val Loss: 24.734776
Epoch 12/2000 | Train Loss: 328.419137		 Val Loss: 24.049530
Epoch 13/2000 | Train Loss: 320.228742		 Val Loss: 23.766991
Epoch 14/2000 | Train Loss: 317.713490		 Val Loss: 22.772242
Epoch 15/2000 | Train Loss: 310.844384		 Val Loss: 22.743896
Epoch 16/2000 | Train Loss: 304.342982		 Val Loss: 22.055496
Epoch 17/2000 | Train Loss: 297.3

In [90]:
model_instance.eval()

with torch.no_grad():
    yys = []
    yyhats = []

    for XX, yy in test_dataloader:
        XX = XX.to(device)
        yys.append(yy.to(device))
        yyhats.append(model_instance(XX).prediction_outputs)

    yyhat = torch.concat(yyhats)
    yy = torch.concat(yys)

    test_loss = loss_fn(yyhat, yy)

/root/anaconda3/envs/TF/lib/python3.12/site-packages/torch/nn/modules/loss.py:634: UserWarning: Using a target size (torch.Size([528, 24, 1])) that is different to the input size (torch.Size([528, 1, 24])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


In [ ]:
mseLoss = torch.nn.MSELoss()
maeLoss = torch.nn.L1Loss()

def smape(yy, yyhat):
    numerator = 100*abs(yy - yyhat)
    denominator = (abs(yy) + abs(yyhat))/2
    smape = torch.mean(numerator / denominator)
    return smape

print(f"test RMSE: {torch.sqrt(mseLoss(yyhat, yy))}")
print(f"test MAE: {maeLoss(yyhat, yy)}")
print(f"test SMAPE: {smape(yy, yyhat)}")

test RMSE: 4.086418628692627
test MAE: 1.8328630924224854
test SMAPE: 2.5656397342681885


/root/anaconda3/envs/TF/lib/python3.12/site-packages/torch/nn/modules/loss.py:132: UserWarning: Using a target size (torch.Size([528, 24, 1])) that is different to the input size (torch.Size([528, 1, 24])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.l1_loss(input, target, reduction=self.reduction)


: 